# What Is a Tool

**Prerequisites:** from `01_foundations`: `05_structured_outputs.ipynb`, `06_function_calling.ipynb`. From `02_agent_runtime`: `02_the_agent_loop.ipynb`

Tool calling already works. Chapter 1 made a call, chapter 2 ran one inside a loop. So this notebook is not about the mechanics.

It is about a problem that only shows up once you write tools regularly. To give a model one tool, you end up writing the same thing three times:

- the Python function that does the work
- a description of that function in JSON, which is what the model reads
- a check that the arguments the model sends back are the right shape

Three pieces, one fact. Python has no idea they are related, so it cannot tell you when they stop agreeing.

This notebook breaks a working tool exactly that way, then fixes it for good: keep the function, and work the other two out from it automatically.

**By the end you'll have:**
- Broken a tool by editing one of the three pieces, and watched the argument check pass right before the call crashed
- Built `args_model()`, which reads a function's parameters and builds the argument check out of them
- Built `Tool` and `@tool` in `tools.py`, where the JSON the model reads is generated on demand instead of typed by hand
- Seen where every line of that generated JSON comes from
- Kept chapter 1's way of writing rules a parameter's type can't express, like "this number must not be zero"
- Checked that a real model accepts the generated JSON and uses it correctly
- Run all of it in chapter 2's ReAct loop without changing anything in `02_agent_runtime`

## Setup

Same as the earlier chapters: `tools.py` sits one folder up, and chapter 1's `llm.py` and chapter 2's runtime come from their own folders.

In [1]:
import json
import sys
from pathlib import Path
from types import SimpleNamespace
from typing import Annotated, Literal

sys.path.append(str(Path.cwd().parent))
sys.path.append(str(Path.cwd().parent.parent / "01_foundations"))
sys.path.append(str(Path.cwd().parent.parent / "02_agent_runtime"))

from pydantic import BaseModel, ValidationError, field_validator, model_validator

from llm import client, DEFAULT_MODEL
from tools import Tool, args_model, tool

## A tool written by hand

Here is a weather tool, written the way chapter 1 wrote its tools. Read it as three separate things that happen to sit near each other in a file:

In [2]:
# 1. the function that does the work
def weather(city, units="celsius"):
    reading = {"tokyo": 18, "oslo": 4, "cairo": 33}.get(city.lower())
    return f"{reading} degrees {units}, clear skies in {city}."

# 2. the schema the model reads
weather_schema = {"type": "function", "function": {
    "name": "weather",
    "description": "Look up the current weather in a city.",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {"type": "string"},
            "units": {"type": "string"},
        },
        "required": ["city"],
    },
}}

# 3. the model that validates what comes back
class WeatherArgs(BaseModel):
    city: str
    units: str = "celsius"

FUNCTIONS = {"weather": weather}
SCHEMAS = {"weather": WeatherArgs}

It works, and it will keep working as long as all three say the same thing.

But they are three ways of stating one fact: *this tool takes a city and some units*. Python sees three unrelated objects. Nothing imports anything, so nothing gets checked against anything.

Now something completely ordinary happens. Someone renames a parameter, because `location` reads better than `city`:

In [3]:
def weather(location, units="celsius"):
    reading = {"tokyo": 18, "oslo": 4, "cairo": 33}.get(location.lower())
    return f"{reading} degrees {units}, clear skies in {location}."

FUNCTIONS = {"weather": weather}

No error. Nothing was linking the three pieces, so there is nothing to break loudly.

The JSON still says `city`, so the model still sends `city`. Chapter 1's `execute_tool_call()` then checks those arguments against `WeatherArgs` — which also still says `city`:

In [4]:
call = SimpleNamespace(id="call_1", function=SimpleNamespace(
    name="weather", arguments='{"city": "Tokyo"}'))

args = SCHEMAS["weather"].model_validate(json.loads(call.function.arguments)).model_dump()
print("validation passed:", args)

try:
    FUNCTIONS["weather"](**args)
except TypeError as e:
    print("the call itself:", type(e).__name__, "-", e)

validation passed: {'city': 'Tokyo', 'units': 'celsius'}
the call itself: TypeError - weather() got an unexpected keyword argument 'city'


**The check passed, and the call crashed anyway.**

This is not a hole in the checking. The check did exactly what it was asked to do. It was asked: *do these arguments match `WeatherArgs`?* They do. Nobody asked whether `WeatherArgs` still matches the `weather` function, because there is nowhere to ask that.

That is the part worth remembering. The check is not watching the other two pieces. It is one of them. It went stale at the same time the JSON did, and because those two still agree with each other, nothing caught the problem before the function ran.

It also breaks in the worst possible place. `TypeError: weather() got an unexpected keyword argument 'city'` travels back to chapter 2's `Act`, which hands it to `DebugThink` to repair. So the model is now being asked to fix its arguments — and it can't. Its arguments were right for the description it was given. It rewrites them, fails again, and spends every retry on a mistake that was ours.

## One object instead of three

The fix is not more checking. It is having one place where the fact lives, and working everything else out from there.

A function's signature is already that place. It has the parameter names, their types, which ones are required, and what the defaults are. Everything the model needs is written there once, and it is the copy that cannot go stale — it *is* the function.

`args_model()` in `tools.py` reads a signature and builds the Pydantic model that checks arguments:

```python
def args_model(function, name=None):
    fields = {}
    for parameter_name, parameter in inspect.signature(function).parameters.items():
        ...
        hint = parameter.annotation
        description = None
        if get_origin(hint) is Annotated:
            hint, *extras = get_args(hint)
            description = next((e for e in extras if isinstance(e, str)), None)
        default = ... if parameter.default is inspect.Parameter.empty else parameter.default
        fields[parameter_name] = (hint, Field(default, description=description))

    return create_model(f"{name or function.__name__}_args",
                        __config__=ConfigDict(extra="forbid"), **fields)
```

`Tool` then keeps the function and that model together. The JSON for the model is a property, so it gets built fresh each time it is asked for rather than stored anywhere:

```python
class Tool:
    def __init__(self, function, name=None, description=None, args=None):
        self.function = function
        self.name = name or function.__name__
        self.description = description or inspect.getdoc(function) or ""
        self.args = args or args_model(function, self.name)
```

`@tool` is all of that in one line:

In [5]:
@tool
def get_weather(
    city: Annotated[str, "The city to look up, e.g. 'Tokyo'"],
    units: Annotated[Literal["celsius", "fahrenheit"], "Temperature scale"] = "celsius",
) -> str:
    """Look up the current weather in a city."""
    reading = {"tokyo": 18, "oslo": 4, "cairo": 33}.get(city.lower())
    if reading is None:
        return f"No weather station for {city}."
    if units == "fahrenheit":
        reading = round(reading * 9 / 5 + 32, 1)
    return f"{reading} degrees {units}, clear skies in {city}."

print(get_weather)
print(get_weather(city="Tokyo"))

<Tool get_weather(city, units)>
18 degrees celsius, clear skies in Tokyo.


`get_weather` is now a `Tool`, not a function — the decorator replaced it, the same way `@property` replaces a method. The original is still there as `get_weather.function` if you ever need it directly.

Calling the `Tool` checks the arguments first, then calls through. So there is no way to reach the function with arguments its own description would have rejected.

Here is what the model actually receives. This JSON description is called the tool's **schema**:

In [6]:
print(json.dumps(get_weather.schema, indent=2))

{
  "type": "function",
  "function": {
    "name": "get_weather",
    "description": "Look up the current weather in a city.",
    "parameters": {
      "additionalProperties": false,
      "properties": {
        "city": {
          "description": "The city to look up, e.g. 'Tokyo'",
          "type": "string"
        },
        "units": {
          "default": "celsius",
          "description": "Temperature scale",
          "enum": [
            "celsius",
            "fahrenheit"
          ],
          "type": "string"
        }
      },
      "required": [
        "city"
      ],
      "type": "object"
    }
  }
}


Every line of that came from somewhere, and none of it was typed twice:

| In the schema | Comes from |
|---|---|
| `"name": "get_weather"` | `function.__name__` |
| `"description"` | the docstring, read by `inspect.getdoc` |
| `"city"`, `"units"` | the parameter names |
| `"type": "string"` | the type written next to each parameter |
| `"description"` on each parameter | the text inside `Annotated[...]` |
| `"enum": ["celsius", "fahrenheit"]` | `Literal[...]` |
| `"default": "celsius"` | the default value in the signature |
| `"required": ["city"]` | having no default |
| `"additionalProperties": false` | `extra="forbid"` |

Two of those are worth more than just saved typing.

**The list of allowed values** (`enum`). `Literal["celsius", "fahrenheit"]` is written once and does two jobs: the model is shown which values it may use, and the argument check rejects anything else. Typed by hand, this is the easiest thing to leave out of the JSON — and the tool then works fine, right up until the model sends `"kelvin"`.

**Parameter descriptions come from `Annotated`, not from the docstring.** The other option was to read them out of a docstring section like `Args:`. That means picking one docstring style and hoping everyone writes that one. Write it Google style when the reader expects NumPy style, and the descriptions quietly disappear from the schema. Nothing errors — the tool just gets harder for the model to use correctly. `Annotated` can't half-work. It is either in the signature or it isn't.

## The same rename, again

Here is the edit that broke everything the first time — `city` becomes `location` — made this time to the generated version:

In [7]:
@tool
def get_weather_renamed(
    location: Annotated[str, "The city to look up, e.g. 'Tokyo'"],
    units: Annotated[Literal["celsius", "fahrenheit"], "Temperature scale"] = "celsius",
) -> str:
    """Look up the current weather in a city."""
    return f"18 degrees {units}, clear skies in {location}."

print("schema parameters:", list(get_weather_renamed.schema["function"]["parameters"]["properties"]))
print("validator fields: ", list(get_weather_renamed.args.model_fields))

schema parameters: ['location', 'units']
validator fields:  ['location', 'units']


Both followed the rename, because neither one is a copy. Both are read off the signature at the moment they are needed.

There is no version of this edit that leaves them disagreeing, which is a much better guarantee than remembering to update three places.

## What the generated check catches

The same Pydantic model that produced the schema is the one that checks what comes back. So anything the schema promises is enforced for free:

In [8]:
for bad in [{"city": "Tokyo", "units": "kelvin"},   # outside the enum
            {"city": "Tokyo", "country": "Japan"},  # an argument that doesn't exist
            {"units": "celsius"}]:                  # missing a required argument
    try:
        get_weather(**bad)
    except ValidationError as e:
        print(f"{bad}\n  -> {e.errors()[0]['msg']}\n")

{'city': 'Tokyo', 'units': 'kelvin'}
  -> Input should be 'celsius' or 'fahrenheit'

{'city': 'Tokyo', 'country': 'Japan'}
  -> Extra inputs are not permitted

{'units': 'celsius'}
  -> Field required



The middle one is a deliberate change from how Pydantic normally behaves.

By default, an argument that was never declared is simply ignored. It gets dropped, and the function runs as if the model had never sent it. So a model asking for the weather in Tokyo, *Japan* gets a perfectly normal-looking answer back, and never finds out that half its request was thrown away.

`extra="forbid"` turns that into an error instead. The model can read it and fix it, and usually does on the next attempt — the same way it corrected a wrong type in chapter 1's notebook 6.

Two other things are rejected when a tool is *defined* rather than when it is called: a parameter with no type, and `*args`/`**kwargs`. In both cases the schema we could generate would describe the function more vaguely than it deserves. Whoever is writing the tool can fix that. The model can't.

In [9]:
def untyped(x):
    return x

def variadic(*items: int):
    return sum(items)

for function in (untyped, variadic):
    try:
        args_model(function)
    except TypeError as e:
        print(e)

untyped: parameter 'x' has no type annotation, so there is nothing to tell the model about it.
variadic: *items cannot be described to a model. Declare the arguments the tool actually takes.


## Rules a signature can't express

Working things out from the signature is the default, not the only option.

Chapter 1's notebook 6 rejected a divisor of zero with a `field_validator`. No type can say that: `0` is a perfectly ordinary number. `Tool` takes a Pydantic model written by hand for exactly this case, and nothing else about it changes:

In [10]:
class DivideArgs(BaseModel):
    a: float
    b: float

    @field_validator("b")
    @classmethod
    def b_must_not_be_zero(cls, v):
        if v == 0:
            raise ValueError("b must not be zero")
        return v

divide = Tool(lambda a, b: a / b, name="divide",
              description="Divide the first number by the second.", args=DivideArgs)

print(json.dumps(divide.schema["function"]["parameters"], indent=2))
print("\n10 / 4 =", divide(a=10, b=4))
try:
    divide(a=10, b=0)
except ValidationError as e:
    print("10 / 0 ->", e.errors()[0]["msg"])

{
  "properties": {
    "a": {
      "type": "number"
    },
    "b": {
      "type": "number"
    }
  },
  "required": [
    "a",
    "b"
  ],
  "type": "object"
}

10 / 4 = 2.5
10 / 0 -> Value error, b must not be zero


Notice what the schema does *not* say. There is no way to write "b must not be zero" in JSON, so the model can still ask for it. The check is not there to stop the model asking. It is there so the answer comes back as a sentence the model can act on, before `divide` runs and raises `ZeroDivisionError` from somewhere deep inside.

Writing the model by hand does bring back the original problem: `DivideArgs` can go stale against the function it checks, exactly like `WeatherArgs` did. That is a real cost, and it is why this is the exception rather than the usual way. It buys the rules a signature genuinely cannot carry, and nothing more.

### Rules that join two fields

`field_validator` handles one value on its own. Plenty of real rules are not about one value.

The common shape is a **categorical field narrowing another one**. Every deployment environment below is a real environment and every strategy is a real strategy — but `recreate` takes the service down while it swaps versions, which is fine in `dev` and not something you do to `production`. Which values of `strategy` are legal depends entirely on what `environment` says.

Watch what the schema manages to say about that:

In [11]:
STRATEGIES = {
    "dev":        {"recreate", "rolling", "blue_green", "canary"},
    "staging":    {"recreate", "rolling", "blue_green", "canary"},
    "production": {"blue_green", "canary"},          # the other two take the service down
}

class DeployArgs(BaseModel):
    environment: Literal["dev", "staging", "production"]
    strategy: Literal["recreate", "rolling", "blue_green", "canary"]

    @model_validator(mode="after")
    def strategy_must_suit_environment(self):
        allowed = STRATEGIES[self.environment]
        if self.strategy not in allowed:
            raise ValueError(f"{self.strategy!r} causes downtime; in {self.environment} "
                             f"use one of {sorted(allowed)}")
        return self

deploy = Tool(lambda environment, strategy: f"deploying to {environment} via {strategy}",
              name="deploy", description="Roll out the service to an environment.",
              args=DeployArgs)

print(json.dumps(deploy.schema["function"]["parameters"]["properties"], indent=2))
print()
print(deploy(environment="dev", strategy="recreate"))
try:
    deploy(environment="production", strategy="recreate")
except ValidationError as e:
    print("production ->", e.errors()[0]["msg"])

{
  "environment": {
    "enum": [
      "dev",
      "staging",
      "production"
    ],
    "type": "string"
  },
  "strategy": {
    "enum": [
      "recreate",
      "rolling",
      "blue_green",
      "canary"
    ],
    "type": "string"
  }
}

deploying to dev via recreate
production -> Value error, 'recreate' causes downtime; in production use one of ['blue_green', 'canary']


**Two independent lists, and nothing joining them.** The model is shown all three environments and all four strategies, and JSON Schema has no way to say *these four, except in production, where only two*. So a model that read its instructions perfectly will still ask for `production` with `recreate`.

That is the same gap `b must not be zero` opened, one step wider. There the schema could not describe a rule about a single value. Here it cannot describe a *relationship* between two, which is the more common case by far — a currency that only some countries accept, a file format that only some compressions apply to, a role that only some permissions come with.

So the rule has to live in the validator. And it has to be a `model_validator`, because of when each kind runs. Pydantic builds an object in three stages:

| | runs on | state |
|---|---|---|
| `model_validator(mode="before")` | the raw dict the model sent | nothing parsed — values are exactly as they arrived |
| `field_validator` | one field at a time, in declaration order | that one field, parsed |
| `model_validator(mode="after")` | `self` | every field parsed and already checked individually |

The line that decides between the two modes: **`after` never runs if any field failed.** So a problem that stops the object being built at all — a field arriving as `""` where an `int` is wanted, two values crammed into one key — has to be fixed in `before`, because there will be no object for `after` to receive. Everything else is `after`, which is nearly everything.

**Could a `field_validator` do it anyway?** Nearly, and the "nearly" is worth seeing. A field validator can take an `info` argument, and `info.data` holds the fields validated *so far*. Here is the same rule written that way, attached to `environment`:

In [12]:
class TryFieldValidator(BaseModel):
    environment: Literal["dev", "staging", "production"]
    strategy: Literal["recreate", "rolling", "blue_green", "canary"]

    @field_validator("environment")
    @classmethod
    def strategy_must_suit_environment(cls, v, info):
        if "strategy" in info.data and info.data["strategy"] not in STRATEGIES[v]:
            raise ValueError("not allowed here")
        return v

print(TryFieldValidator(environment="production", strategy="recreate"), " <- never fired")

environment='production' strategy='recreate'  <- never fired


**Accepted, silently.** Fields are validated in the order they are declared, and `environment` comes first — so when its validator ran, `strategy` had not been touched yet. `info.data` was empty, `"strategy" in info.data` was false, and the check skipped itself without a word.

Move the rule to `strategy` and it works, because by then `environment` is done. Which is the problem: the rule is only correct while the two fields stay in that order, and nothing in the file says so. Reorder them a year from now for readability and your validation quietly stops existing.

Two more ways it goes quiet, for the same reason:

- **If the other field fails to parse, it never reaches `info.data`.** Send `environment="prod"` (not in the `Literal`) and the guard is false again — you get one error about the bad environment and no answer about whether the pair made sense.
- **A field validator returns one field's value.** Even seeing both, it has no way to change the other one.

`model_validator(mode="after")` has none of these. It runs once, at the end, holding the whole object. Declaration order stops mattering, every field is present, and there is nothing to guard against. That is the difference between the two decorators — not preference: one runs *per field, mid-build, with partial information*, the other runs *once, at the end, with all of it*.

### Raise, don't repair

A `model_validator` can also return a corrected `self` instead of raising. It is worth saying plainly why this notebook doesn't.

**Most of the time there is no correction to make.** Given `production` + `recreate`, there are two ways to make it valid and they mean opposite things: change the strategy and you deploy differently from what was asked; change the environment and you deploy to the wrong place. Nothing in the input says which was intended. A range that runs backwards is the one shape where a fix *looks* obvious — just swap them — and even there the swap is a guess, because a model that returned two dates in the wrong order most likely misread the question, and swapping hides the evidence.

**And there is already a pattern for a bad argument.** It is the one this chapter opened with: raise, let the message reach `DebugThink`, let the model try again knowing exactly what was wrong. `extract()` in `llm.py` does the same thing for structured output. A second pattern buys one saved round trip and costs three things — fix-up logic written by hand for every rule, a change nobody can see in the output, and a guess about intent baked into code where no one will look at it again.

**The error is doing real work here, too.** Look at what it says: *`'recreate'` causes downtime; in production use one of `['blue_green', 'canary']`*. That is the rule the schema could not carry, arriving at the model at the only moment it could — after the mistake, in a message it can read. The retry is not a fresh guess. It is the first time the model has been told.

The narrow case where repair is still right is **normalising, not correcting**: stripping whitespace, lowercasing a city name, folding `"Celsius"` into `"celsius"`. Those canonicalise a value that was already unambiguous. Deciding what somebody meant is a different act, and it belongs to the model, which gets another turn anyway.

## Does a real model accept it?

Everything so far has run offline. Two things are left to check: that Groq accepts a schema generated by Pydantic, and that the model can actually use it. The list of allowed values matters most, since it is the part nobody typed:

In [13]:
response = client.chat.completions.create(
    model=DEFAULT_MODEL,
    messages=[{"role": "user", "content": "What's the weather in Cairo? Use fahrenheit."}],
    tools=[get_weather.schema],
)
tool_call = response.choices[0].message.tool_calls[0]
print("the model asked for:", tool_call.function.name, tool_call.function.arguments)
print("result:", get_weather(**json.loads(tool_call.function.arguments)))

the model asked for: get_weather {"city":"Cairo","units":"fahrenheit"}
result: 91.4 degrees fahrenheit, clear skies in Cairo.


Accepted, and both arguments are right. The model could only have spelled `units` that way by reading the allowed values Pydantic generated from the `Literal`.

## Into chapter 2's loop, unchanged

Chapter 2's `Act` wants a dict of functions and a dict of argument checks. Its `Think` wants a list of schemas. A `Tool` carries all three of those, so the wiring is three one-line comprehensions — and nothing in `02_agent_runtime` has to change:

In [14]:
from chat import Conversation
from runtime import Runtime
from observe import Observe
from think import Think
from act import Act

@tool
def sunset_time(city: Annotated[str, "The city to look up"]) -> str:
    """The local time the sun sets today in a city."""
    at = {"tokyo": "18:12", "oslo": "21:47", "cairo": "17:55"}.get(city.lower())
    return f"The sun sets at {at} in {city}." if at else f"No sunset data for {city}."

TOOLS = [get_weather, sunset_time]

schemas_for_model = [t.schema for t in TOOLS]        # what Think sends the API
functions = {t.name: t.function for t in TOOLS}      # what Act calls
schemas = {t.name: t.args for t in TOOLS}            # what Act validates against

runtime = Runtime(
    loop=[
        Observe(event="What's the weather in Oslo, and when does the sun set there?"),
        Think(tools=schemas_for_model, allow_fork=False),
        Act(functions=functions, schemas=schemas),
        Observe(),
    ],
    repeat_from=1,
)

conversation = runtime.run(Conversation())
for m in conversation.messages:
    print(f"- {m['role']:9} {m['content']!r}")

- user      "What's the weather in Oslo, and when does the sun set there?"
- decision  '{"city":"Oslo","units":"celsius"}'
- tool      'The current weather in Oslo is 4 degrees celsius with clear skies.'
- decision  '{"city":"Oslo"}'
- tool      'The sun sets at 21:47 in Oslo.'
- decision  'The current weather in Oslo is 4 degrees celsius with clear skies. The sun sets at 21:47 in Oslo.'
- assistant 'The current weather in Oslo is 4 degrees celsius with clear skies. The sun sets at 21:47 in Oslo.'


`Act` runs `schemas[name].model_validate(args)`, the same line it has always run — except the model it checks against is one nobody wrote.

`functions` holds `t.function`, the plain undecorated function, because `execute_tool_call()` already checks the arguments. Going through `Tool.__call__` would just check them twice.

Those three lines are also a fair summary of where this notebook stops. Three dicts, built by hand from one list, each needed by a different part of the loop. With two tools that is fine. With thirty it is a list to keep in sync — the same problem this notebook opened with, one level up.

## What you built

✓ Reproduced a real failure: a renamed parameter, an argument check that passed, and a crash one line later — because the check was one of the pieces that went stale, not something watching the others

✓ Followed that failure into chapter 2, where `DebugThink` asks the model to repair arguments that were already correct, so every retry is wasted

✓ Built `args_model()`, which reads a function's parameters and builds the argument check from them: names, types, defaults, which ones are required, and lists of allowed values

✓ Built `Tool` and `@tool` in `tools.py` — one object holding the function, generating the schema on demand and checking arguments against the same source, so the rename that broke the hand-written version can't break this one

✓ Chose `Annotated[str, "..."]` over reading descriptions out of docstrings, because a docstring in the wrong style fails silently and a missing `Annotated` doesn't

✓ Used `extra="forbid"`, so an argument the tool never declared becomes an error the model can fix instead of something quietly dropped

✓ Kept chapter 1's `field_validator` for rules a signature can't express, added `model_validator` for rules that join two fields — an enum whose legal values depend on another field, which JSON Schema cannot express at all — and said plainly what both cost: a check written by hand can go stale again

✓ Kept one pattern for a bad argument — raise, and let `DebugThink` retry with the real message — rather than repairing it quietly in a validator, because a repair encodes a guess about intent and hides the evidence that the prompt needs work

✓ Confirmed Groq accepts a generated schema, and that the model used the allowed values it was never explicitly told about

✓ Ran all of it inside chapter 2's ReAct loop with `Think(allow_fork=False)`, changing nothing in `02_agent_runtime`

**Next:** `02_the_registry.ipynb` — the three dicts built by hand at the end of this notebook become one object that owns them. It also brings up the questions that only appear once there are more tools than fit comfortably in a prompt: two tools wanting the same name, and giving one agent a few of the tools instead of all of them.